In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/032026/Data_OOT/MEDS_MDP/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,45,1956-05-29 00:00:00,DOB,NaN
1,45,2018-11-27 13:30:00,P/UXRG40,NaN
2,45,2019-06-14 09:59:00,P/UXRC45,NaN
3,45,2020-01-25 00:00:00,D/DM768A,NaN
4,45,2020-02-25 09:36:00,P/ZZ0150,NaN
5,45,2020-02-25 10:45:00,P/UXRG30,NaN
6,45,2020-03-12 10:30:00,P/UXRF40,NaN
7,45,2020-06-03 10:16:00,P/WKDMAXXXX,NaN
8,45,2020-06-16 14:02:00,P/AAF22,NaN
9,45,2020-06-19 12:25:00,P/UXMF40,NaN


In [2]:
df.head(20)

,subject_id,time,code,numeric_value
0,45,1956-05-29 00:00:00,DOB,NaN
1,45,2018-11-27 13:30:00,P/UXRG40,NaN
2,45,2019-06-14 09:59:00,P/UXRC45,NaN
3,45,2020-01-25 00:00:00,D/DM768A,NaN
4,45,2020-02-25 09:36:00,P/ZZ0150,NaN
5,45,2020-02-25 10:45:00,P/UXRG30,NaN
6,45,2020-03-12 10:30:00,P/UXRF40,NaN
7,45,2020-06-03 10:16:00,P/WKDMAXXXX,NaN
8,45,2020-06-16 14:02:00,P/AAF22,NaN
9,45,2020-06-19 12:25:00,P/UXMF40,NaN


In [5]:
len(df)

587671613

In [6]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 2218028
The patients has M-medication Codes: 1511054
The patients has D-diagnosis Codes: 2217421
The patients has P-Procedure Codes: 2126853
The patients has S-SKS Codes: 0


In [2]:
subject_counts = df['subject_id'].value_counts()

In [5]:
subject_counts

921542     82176
106        63833
698589     63448
2016077    60882
954669     58832
           ...  
1434427        2
1831164        2
1069671        2
1703144        2
296240         2
Name: subject_id, Length: 2218028, dtype: int64

In [6]:
s_Num = df[df['code'].str.startswith('P/', na=False)]

In [7]:
s_Num

,subject_id,time,code,numeric_value
1,45,2018-11-27 13:30:00,P/UXRG40,NaN
2,45,2019-06-14 09:59:00,P/UXRC45,NaN
4,45,2020-02-25 09:36:00,P/ZZ0150,NaN
5,45,2020-02-25 10:45:00,P/UXRG30,NaN
6,45,2020-03-12 10:30:00,P/UXRF40,NaN
...,...,...,...,...
587671607,2218014,2024-05-16 09:56:00,P/UXUD88D,NaN
587671608,2218014,2024-05-16 09:56:00,P/UXUD88F,NaN
587671609,2218014,2024-05-16 10:26:00,P/BKUA1C,NaN
587671610,2218014,2024-05-31 07:52:00,P/ZZ0151,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Srugery code: ", only_p_ids_to_exclude)


Number of patients with only Srugery code:  []


In [3]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

921542     77794
698589     61367
106        60339
2016077    58273
954669     55553
           ...  
1426563        2
55620          2
239646         2
1626868        2
1436606        2
Name: subject_id, Length: 2218028, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
7,1291159,56356,50022,6334
31,714050,43018,37691,5327
5,1964088,57193,52288,4905
8948,1887674,4652,66,4586
0,921542,82176,77794,4382
...,...,...,...,...
2149777,1495753,7,7,0
2149775,614394,7,7,0
2149774,1174687,7,7,0
2149773,2105107,7,7,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 91175


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


      subject_id  original_count  new_count  difference  abs_diff
7        1291159           56356      50022        6334      6334
31        714050           43018      37691        5327      5327
5        1964088           57193      52288        4905      4905
8948     1887674            4652         66        4586      4586
0         921542           82176      77794        4382      4382
16       1169555           49087      45012        4075      4075
49          9512           36671      33013        3658      3658
218      2109519           20292      16667        3625      3625
1            106           63833      60339        3494      3494
4         954669           58832      55553        3279      3279


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [22]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MD codes'
    }
)


In [23]:
lowest_new_count_patients

,subject_id,MDPS codes,MD codes,difference,abs_diff
1681151,356913,31,2,29,29
1810362,1181764,24,2,22,22
1849155,1896771,22,2,20,20
1870477,816167,21,2,19,19
1886339,2076088,21,2,19,19
...,...,...,...,...,...
2208430,2087106,3,2,1,1
2204191,715600,3,2,1,1
2199938,1144261,3,2,1,1
2198459,1328943,3,2,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [24]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

2218028

In [25]:
len(df_filtered)

482176096

In [4]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('P/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


kept rows MDP: 482176096  / total: 587671613


In [5]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 482176096


In [6]:
import numpy as np
import os

N_SHARDS = 45    #45 for Whole # 36 when we have split
OUT_DIR = "./_TrainMDPS_withoutSP_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 45 parquet files into ./_TrainMDPS_withoutSP_sharded


In [7]:
import pyarrow.parquet as pq

OUT_DIR = "./_TrainMDPS_withoutSP_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 10643760
1.parquet rows: 10675809
10.parquet rows: 10782066
11.parquet rows: 10810081
12.parquet rows: 10580931
13.parquet rows: 10864069
14.parquet rows: 11062596
15.parquet rows: 10555104
16.parquet rows: 10875764
17.parquet rows: 10794132
18.parquet rows: 10742792
19.parquet rows: 11011089
2.parquet rows: 10650135
20.parquet rows: 10451229
21.parquet rows: 10575943
22.parquet rows: 10807067
23.parquet rows: 10901640
24.parquet rows: 10667284
25.parquet rows: 10831502
26.parquet rows: 10589414
27.parquet rows: 10844410
28.parquet rows: 10623027
29.parquet rows: 10683499
3.parquet rows: 10585535
30.parquet rows: 10701026
31.parquet rows: 10754490
32.parquet rows: 10963943
33.parquet rows: 10676279
34.parquet rows: 10320042
35.parquet rows: 10947739
36.parquet rows: 10600134
37.parquet rows: 10875239
38.parquet rows: 10549385
39.parquet rows: 10656185
4.parquet rows: 10496521
40.parquet rows: 10746289
41.parquet rows: 10694327
42.parquet rows: 10956754
43.parquet rows: 

In [8]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_TrainMDPS_withoutSP_sharded"
DST_PREFIX = "Zahra/032026/Data_OOT/MEDS_MD/data/train"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 45
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutSP_sharded' to 'Zahra/032026/Data_OOT/MEDS_MD/data/train'
Copying 45 files with concurrency set to 6
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutSP_sharded/12.parquet, file 1 out of 45. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/032026/Data_OOT/MEDS_MD/data/train/12.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_OOT/corebehrt/Myconfigs/CreateIdenticalData/_TrainMDPS_withoutSP_sharded/11.parquet, file 2 out of 45. Destination path: https://

In [9]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:50])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 45
['/0.parquet', '/1.parquet', '/10.parquet', '/11.parquet', '/12.parquet', '/13.parquet', '/14.parquet', '/15.parquet', '/16.parquet', '/17.parquet', '/18.parquet', '/19.parquet', '/2.parquet', '/20.parquet', '/21.parquet', '/22.parquet', '/23.parquet', '/24.parquet', '/25.parquet', '/26.parquet', '/27.parquet', '/28.parquet', '/29.parquet', '/3.parquet', '/30.parquet', '/31.parquet', '/32.parquet', '/33.parquet', '/34.parquet', '/35.parquet', '/36.parquet', '/37.parquet', '/38.parquet', '/39.parquet', '/4.parquet', '/40.parquet', '/41.parquet', '/42.parquet', '/43.parquet', '/44.parquet', '/5.parquet', '/6.parquet', '/7.parquet', '/8.parquet', '/9.parquet']
